# NBA API

In [1]:
# Tools
import pandas as pd
import numpy as np
import time
import re
from typing import Tuple, Optional

from nba_api.stats.endpoints import teamgamelogs, leaguestandingsv3, leaguestandings

## 1.1 Data Retrieval

In [2]:
# Define the seasons from 2014-15 to 2023-24
seasons = ['2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']

In [3]:
# Set up sleep time to avoid api call explosion
def cool_down(delay=5):
    print(f"Cooling down for {delay} seconds...")
    time.sleep(delay)

### League Standings

In [4]:
standings_all = []

for season in seasons:
    print(f"Retrieving League Standings (prefer V3): {season}")
    try:
        # --- Try V3 first ---
        standings_v3 = leaguestandingsv3.LeagueStandingsV3(season=season)
        # retrieve using generic data frame getter
        df = standings_v3.get_data_frames()[0]
        df["SEASON"] = season
        df["SOURCE"] = "V3"
        print(f"   → Success using LeagueStandingsV3 for {season}")
    except Exception as e_v3:
        print(f"LeagueStandingsV3 failed for {season}: {e_v3}")
        try:
            # --- Fallback to classic endpoint ---
            standings_v1 = leaguestandings.LeagueStandings(season=season)
            df = standings_v1.get_data_frames()[0]
            df["SEASON"] = season
            df["SOURCE"] = "V1"
            print(f"   → Fallback success using LeagueStandings for {season}")
        except Exception as e_v1:
            print(f"Unable to retrieve any standings for {season}: {e_v1}")
            df = pd.DataFrame()

    if not df.empty:
        standings_all.append(df)
    cool_down(5)

# Combine results
standings_df = pd.concat(standings_all, ignore_index=True)
standings_df.to_csv("data/league_standings_combined.csv", index=False)
print(f"Saved combined standings: data/league_standings_combined.csv")


Retrieving League Standings (prefer V3): 2014-15
   → Success using LeagueStandingsV3 for 2014-15
Cooling down for 5 seconds...
Retrieving League Standings (prefer V3): 2015-16
   → Success using LeagueStandingsV3 for 2015-16
Cooling down for 5 seconds...
Retrieving League Standings (prefer V3): 2016-17
   → Success using LeagueStandingsV3 for 2016-17
Cooling down for 5 seconds...
Retrieving League Standings (prefer V3): 2017-18
   → Success using LeagueStandingsV3 for 2017-18
Cooling down for 5 seconds...
Retrieving League Standings (prefer V3): 2018-19
   → Success using LeagueStandingsV3 for 2018-19
Cooling down for 5 seconds...
Retrieving League Standings (prefer V3): 2019-20
   → Success using LeagueStandingsV3 for 2019-20
Cooling down for 5 seconds...
Retrieving League Standings (prefer V3): 2020-21
   → Success using LeagueStandingsV3 for 2020-21
Cooling down for 5 seconds...
Retrieving League Standings (prefer V3): 2021-22
   → Success using LeagueStandingsV3 for 2021-22
Coolin

### Team Game Logs

In [5]:
# TeamGameLogs and Schedule for seasons 2014-15 to 2024-25
teamlogs_dfs = []
for season in seasons:
    try:
        logs = teamgamelogs.TeamGameLogs(league_id_nullable='00',season_nullable=season)
        df = logs.get_data_frames()[0]
        df["SEASON"] = season
        teamlogs_dfs.append(df)
        time.sleep(2)
    except Exception as e:
        print(f"Failed to fetch {season}: {e}")

teamlogs_dfs = pd.concat(teamlogs_dfs, ignore_index=True)
display(teamlogs_dfs)

,SEASON_YEAR,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,AVAILABLE_FLAG,SEASON
0,2014-15,1610612744,GSW,Golden State Warriors,0041400406,2015-06-16T00:00:00,GSW @ CLE,W,48.0,37,...,184,326,1451,2221,2572,294,890,751,1.0,2014-15
1,2014-15,1610612739,CLE,Cleveland Cavaliers,0041400406,2015-06-16T00:00:00,CLE vs. GSW,L,48.0,32,...,2364,2684,404,945,2463,213,1658,2028,1.0,2014-15
2,2014-15,1610612744,GSW,Golden State Warriors,0041400405,2015-06-14T00:00:00,GSW vs. CLE,W,48.0,36,...,1978,1487,2357,945,2316,150,984,431,1.0,2014-15
3,2014-15,1610612739,CLE,Cleveland Cavaliers,0041400405,2015-06-14T00:00:00,CLE @ GSW,L,48.0,32,...,1212,535,1451,218,2653,404,2166,2386,1.0,2014-15
4,2014-15,1610612744,GSW,Golden State Warriors,0041400404,2015-06-11T00:00:00,GSW @ CLE,W,48.0,36,...,44,2201,1004,509,1462,1666,1065,155,1.0,2014-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30243,2024-25,1610612750,MIN,Minnesota Timberwolves,0012400003,2024-10-04T00:00:00,MIN @ LAL,W,48.0,47,...,1773,166,1944,2767,1381,934,531,359,1.0,2024-25
30244,2024-25,1610612738,BOS,Boston Celtics,0012400001,2024-10-04T00:00:00,BOS @ DEN,W,48.0,37,...,2189,166,224,186,2421,273,1866,1114,1.0,2024-25
30245,2024-25,1610612743,DEN,Denver Nuggets,0012400001,2024-10-04T00:00:00,DEN @ BOS,L,48.0,37,...,2746,627,2359,2398,2421,273,2173,1621,1.0,2024-25
30246,2024-25,1610612747,LAL,Los Angeles Lakers,0012400003,2024-10-04T00:00:00,LAL vs. MIN,L,48.0,38,...,2575,915,18,445,1639,1165,1866,2415,1.0,2024-25


## Output DF to CSV files

In [6]:
# Save Team Game Logs
teamlogs_dfs.to_csv("data/team_game_logs.csv", index=False)

# 1. BT-Model

In [7]:
FEATURE_CONFIG = {
    "recent_N": 10,          # window size for recent margin/win%
    "rest_clip": 5,          # clip rest days to [0, rest_clip]        # clip homestand/road-trip length
}

## BT-Model Data-preprocessing

In [8]:
def normalize_abbr(abbr: str) -> str:
    if abbr is None or not isinstance(abbr, str):
        return abbr
    a = abbr.strip().upper()
    mapping = {
        "CHA": "CHA", "CHO": "CHA",
        "BRK": "BKN", "BKN": "BKN",
        "PHO": "PHX", "PHX": "PHX",
        "SAN": "SAS", "SAS": "SAS",
        "NOP": "NOP", "NOH": "NOP",
        "NYK": "NYK", "GSW": "GSW", "LAL": "LAL", "LAC": "LAC",
        "UTA": "UTA", "OKC": "OKC", "MIN": "MIN", "MIL": "MIL",
        "MIA": "MIA", "ORL": "ORL", "ATL": "ATL", "BOS": "BOS",
        "CHI": "CHI", "CLE": "CLE", "DAL": "DAL", "DEN": "DEN",
        "DET": "DET", "HOU": "HOU", "IND": "IND", "MEM": "MEM",
        "POR": "POR", "SAC": "SAC", "TOR": "TOR", "WAS": "WAS",
        "PHI": "PHI",
    }
    return mapping.get(a, a)

def parse_matchup_return_abbrs(matchup: str) -> Tuple[Optional[str], Optional[str]]:
    if not isinstance(matchup, str):
        return None, None
    m = re.sub(r"\s+", " ", matchup.strip())
    if " vs. " in m:
        lhs, rhs = m.split(" vs. ")
        return normalize_abbr(lhs), normalize_abbr(rhs)
    if " @ " in m:
        lhs, rhs = m.split(" @ ")
        return normalize_abbr(rhs), normalize_abbr(lhs)  # rhs is home
    parts = m.split()
    if len(parts) >= 3 and parts[1] in {"vs.", "@"}:
        if parts[1] == "vs.":
            return normalize_abbr(parts[0]), normalize_abbr(parts[2])
        else:
            return normalize_abbr(parts[2]), normalize_abbr(parts[0])
    return None, None

def coalesce_series(*series_list):
    out = series_list[0].copy()
    for s in series_list[1:]:
        out = out.where(out.notna(), s)
    return out

def seasonid_to_text(x):
    try:
        y = int(str(x)[-4:])
        return f"{y}-{str((y+1))[-2:]}"
    except Exception:
        return np.nan

def normalize_season_str(x):
    s = str(x)
    if pd.isna(x) or s.strip() == "" or s.lower() == "nan":
        return np.nan
    s = s.strip()
    if re.fullmatch(r"\d{4}-\d{2}", s):
        return s
    m = re.fullmatch(r"(\d{4})-(\d{4})", s)
    if m:
        start = int(m.group(1))
        return f"{start}-{str(start+1)[-2:]}"
    m = re.fullmatch(r"(\d{4})(\d{2})", s)
    if m:
        start = int(m.group(1))
        return f"{start}-{str(start+1)[-2:]}"
    m = re.fullmatch(r"(\d{4})", s)
    if m:
        start = int(m.group(1))
        return f"{start}-{str(start+1)[-2:]}"
    return np.nan

def to_season_from_date(dt):
    dt = pd.to_datetime(dt, errors="coerce")
    if pd.isna(dt):
        return np.nan
    y = dt.year
    start = y if dt.month >= 8 else y - 1
    return f"{start}-{str(start+1)[-2:]}"

### Load data

In [9]:
# Load inputs
try:
    seasons
except NameError:
    raise NameError("Please define seasons")

tg_log = pd.read_csv('./data/team_game_logs.csv')

### BT-Model Dataset Build

In [10]:
def build_bt_df(raw: pd.DataFrame, keep_seasons=None) -> pd.DataFrame:
    raw = raw.copy()

    # --- Season ---
    if "SEASON_YEAR" in raw.columns:
        raw["season"] = raw["SEASON_YEAR"].astype(str).map(normalize_season_str)
    elif "SEASON" in raw.columns:
        raw["season"] = raw["SEASON"].astype(str).map(normalize_season_str)
    elif "GAME_DATE" in raw.columns:
        raw["season"] = raw["GAME_DATE"].map(to_season_from_date)
    else:
        raw["season"] = pd.NA

    # Filter seasons
    if keep_seasons is not None:
        raw = raw[raw["season"].isin(keep_seasons)].copy()

    # --- Basic cleanup ---
    raw["GAME_ID"] = raw["GAME_ID"].astype(str)
    raw["TEAM_ABBREVIATION_NORM"] = raw["TEAM_ABBREVIATION"].astype(str).map(normalize_abbr)

    # --- Identify home team ---
    raw["isHome"] = raw["MATCHUP"].astype(str).str.contains(" vs. ", na=False)

    # Keep ONLY home rows → guarantees 1 row per game
    home_df = raw[raw["isHome"]].copy()

    # --- Parse opponent (away team) ---
    def get_away(row):
        matchup = str(row["MATCHUP"])
        if " vs. " in matchup:
            return normalize_abbr(matchup.split(" vs. ")[1])
        return None

    home_df["home_abbr"] = home_df["TEAM_ABBREVIATION_NORM"]
    home_df["away_abbr"] = home_df.apply(get_away, axis=1)

    # --- Determine home_win ---
    if "WL" in home_df.columns:
        home_df["home_win"] = (home_df["WL"] == "W").astype(int)
    elif "PLUS_MINUS" in home_df.columns:
        home_df["home_win"] = (pd.to_numeric(home_df["PLUS_MINUS"], errors="coerce") > 0).astype(int)
    else:
        home_df["home_win"] = np.nan

    # --- Final selection ---
    cols = ["season", "GAME_ID", "home_abbr", "away_abbr", "home_win"]
    if "GAME_DATE" in home_df.columns:
        cols.append("GAME_DATE")

    bt = home_df[cols].copy()

    # --- Clean ---
    bt = bt.dropna(subset=["season", "GAME_ID", "home_abbr", "away_abbr", "home_win"])
    bt = bt.drop_duplicates(subset=["GAME_ID"]).reset_index(drop=True)

    if "GAME_DATE" in bt.columns:
        bt["GAME_DATE"] = pd.to_datetime(bt["GAME_DATE"], errors="coerce")
        bt = bt.dropna(subset=["GAME_DATE"])

    return bt

In [11]:
bt_core = build_bt_df(tg_log, keep_seasons=seasons)
bt_core.head()

,season,GAME_ID,home_abbr,away_abbr,home_win,GAME_DATE
0,2014-15,41400406,CLE,GSW,0,2015-06-16
1,2014-15,41400405,GSW,CLE,1,2015-06-14
2,2014-15,41400404,CLE,GSW,0,2015-06-11
3,2014-15,41400403,CLE,GSW,1,2015-06-09
4,2014-15,41400402,GSW,CLE,0,2015-06-07


## BT-Model Data Post-processing

### Build per-team chronological schedule with pre-game rolling and situational features

In [13]:
def build_team_schedule_features(tg: pd.DataFrame, recent_N: int, rest_clip: int):
    g = tg.copy()

    # Normalize basics
    g["TEAM_ABBREVIATION_NORM"] = g["TEAM_ABBREVIATION"].astype(str).map(normalize_abbr)
    # Possibility: Some datasets label opponent implicitly in MATCHUP; we’ll compute outcome and margin from available cols
    # Build margin, win flag from WL if available, else PLUS_MINUS
    if "WL" in g.columns:
        g["win_flag"] = (g["WL"] == "W").astype(int)
    else:
        if "PLUS_MINUS" in g.columns:
            g["win_flag"] = (pd.to_numeric(g["PLUS_MINUS"], errors="coerce") > 0).astype(int)
        else:
            g["win_flag"] = np.nan

    # Points columns
    if "PTS" in g.columns:
        g["pts_for"] = pd.to_numeric(g["PTS"], errors="coerce")
    else:
        g["pts_for"] = np.nan

    # We need opponent points to compute margin; if not present, PLUS_MINUS can serve
    if "PLUS_MINUS" in g.columns and g["PLUS_MINUS"].notna().any() and g["pts_for"].notna().any():
        g["margin"] = pd.to_numeric(g["PLUS_MINUS"], errors="coerce")
        # Optional: infer opp points when margin present
        # g["pts_against"] = g["pts_for"] - g["margin"]
    else:
        # As fallback, set margin NaN; later rolling margin could be NaN early
        g["margin"] = np.nan

    # Home flag from MATCHUP
    g["is_home"] = g["MATCHUP"].astype(str).str.contains(" vs. ", na=False)

    # Dates & seasons
    g["GAME_DATE"] = pd.to_datetime(g["GAME_DATE"], errors="coerce")
    g["season"] = g.get("SEASON_YEAR", g.get("SEASON", None))
    if g["season"].isna().any() or g["season"].dtype != object:
        g["season"] = g["GAME_DATE"].map(to_season_from_date)
    else:
        g["season"] = g["season"].astype(str).map(normalize_season_str)

    # Sort per team-season by date
    g = g.sort_values(["TEAM_ABBREVIATION_NORM", "season", "GAME_DATE"]).reset_index(drop=True)

    # Compute rest
    def compute_team_feats(df):
        df = df.sort_values("GAME_DATE").copy()
        team = df["TEAM_ABBREVIATION_NORM"].iloc[0] if "TEAM_ABBREVIATION_NORM" in df.columns else None
        season = df["season"].iloc[0] if "season" in df.columns else None

        df["TEAM_ABBREVIATION_NORM"] = team
        df["season"] = season
        df["prev_date"] = df["GAME_DATE"].shift(1)
        df["rest_days"] = (df["GAME_DATE"] - df["prev_date"]).dt.days - 1  # gap days between games
        df["rest_days"] = df["rest_days"].clip(lower=0, upper=rest_clip).fillna(rest_clip)
        df["b2b"] = (df["rest_days"] == 0).astype(int)

        # Rolling counts for windows
        df["game_num"] = np.arange(1, len(df)+1)

        # Recent form (pre-game rolling)
        df["rolling_margin"] = df["margin"].rolling(window=recent_N, min_periods=1).mean().shift(1)
        df["rolling_win_pct"] = df["win_flag"].rolling(window=recent_N, min_periods=1).mean().shift(1)

        # Season-to-date points per game for and against (pre-game)
        df["cum_pts_for"] = df["pts_for"].expanding().sum().shift(1)
        df["cum_games"] = (~df["pts_for"].isna()).expanding().sum().shift(1)
        df["pts_pg_to_date"] = df["cum_pts_for"] / df["cum_games"]

        # If margin available, infer opp points cumulative (optional)
        if "margin" in df.columns and df["margin"].notna().any() and df["pts_for"].notna().any():
            df["pts_against"] = df["pts_for"] - df["margin"]
            df["cum_pts_against"] = df["pts_against"].expanding().sum().shift(1)
            df["opp_pts_pg_to_date"] = df["cum_pts_against"] / df["cum_games"]
            df["diff_pts_pg_to_date"] = df["pts_pg_to_date"] - df["opp_pts_pg_to_date"]
        else:
            df["opp_pts_pg_to_date"] = np.nan
            df["diff_pts_pg_to_date"] = np.nan

        # Home/road split win% to date
        df["home_games_cum"] = df["is_home"].astype(int).expanding().sum().shift(1)
        df["home_wins_cum"] = (df["is_home"] & (df["win_flag"]==1)).astype(int).expanding().sum().shift(1)
        df["home_win_pct_to_date"] = np.where(df["home_games_cum"]>0, df["home_wins_cum"]/df["home_games_cum"], np.nan)

        df["road_games_cum"] = ((~df["is_home"]).astype(int)).expanding().sum().shift(1)
        df["road_wins_cum"] = ((~df["is_home"]) & (df["win_flag"]==1)).astype(int).expanding().sum().shift(1)
        df["road_win_pct_to_date"] = np.where(df["road_games_cum"]>0, df["road_wins_cum"]/df["road_games_cum"], np.nan)

        # Keep minimal columns for merge
        keep_cols = [
            "TEAM_ABBREVIATION_NORM","season","GAME_ID","GAME_DATE","is_home","win_flag",
            "rest_days","b2b","rolling_margin","rolling_win_pct",
            "pts_pg_to_date","opp_pts_pg_to_date","diff_pts_pg_to_date",
            "home_win_pct_to_date","road_win_pct_to_date"
        ]
        return df[keep_cols]

    feat = (
        g.groupby(["TEAM_ABBREVIATION_NORM","season"], group_keys=False)
         .apply(compute_team_feats)
         .reset_index(drop=True)
    )
    return feat

team_feats = build_team_schedule_features(
    tg=tg_log,
    recent_N=FEATURE_CONFIG["recent_N"],
    rest_clip=FEATURE_CONFIG["rest_clip"]
)

### Normalize input

In [14]:
bt_core = bt_core.copy()
bt_core["GAME_ID"] = bt_core["GAME_ID"].astype(str)
bt_core["season"] = bt_core["season"].astype(str).map(normalize_season_str)
bt_core["home_abbr"] = bt_core["home_abbr"].astype(str).map(normalize_abbr)
bt_core["away_abbr"] = bt_core["away_abbr"].astype(str).map(normalize_abbr)

team_feats = team_feats.copy()

# Ensure team abbreviation column exists and is normalized
if "TEAM_ABBREVIATION_NORM" not in team_feats.columns:
    # If you only have TEAM_ABBREVIATION, normalize that
    if "TEAM_ABBREVIATION" in team_feats.columns:
        team_feats["TEAM_ABBREVIATION_NORM"] = team_feats["TEAM_ABBREVIATION"].astype(str).map(normalize_abbr)
    else:
        raise KeyError("team_feats must contain TEAM_ABBREVIATION_NORM or TEAM_ABBREVIATION")

team_feats["TEAM_ABBREVIATION_NORM"] = team_feats["TEAM_ABBREVIATION_NORM"].astype(str).map(normalize_abbr)
team_feats["GAME_ID"] = team_feats["GAME_ID"].astype(str)
team_feats["season"] = team_feats["season"].astype(str).map(normalize_season_str)

# Drop rows with missing keys to avoid NaN abbrs in key set
team_feats = team_feats.dropna(subset=["TEAM_ABBREVIATION_NORM", "season", "GAME_ID"]).copy()

### Build per-game context by merging home and away team features

In [15]:
# Build home/away feature views
home_tf = team_feats.rename(columns={
    "TEAM_ABBREVIATION_NORM":"home_abbr",
    "rest_days":"home_rest_days",
    "b2b":"home_b2b",
    "rolling_margin":"home_recent_margin",
    "rolling_win_pct":"home_recent_win_pct",
    "pts_pg_to_date":"home_pts_pg_to_date",
    "opp_pts_pg_to_date":"home_opp_pts_pg_to_date",
    "diff_pts_pg_to_date":"home_diff_pts_pg_to_date",
    "home_win_pct_to_date":"home_home_win_pct_to_date",
    "road_win_pct_to_date":"home_road_win_pct_to_date",
})
away_tf = team_feats.rename(columns={
    "TEAM_ABBREVIATION_NORM":"away_abbr",
    "rest_days":"away_rest_days",
    "b2b":"away_b2b",
    "rolling_margin":"away_recent_margin",
    "rolling_win_pct":"away_recent_win_pct",
    "pts_pg_to_date":"away_pts_pg_to_date",
    "opp_pts_pg_to_date":"away_opp_pts_pg_to_date",
    "diff_pts_pg_to_date":"away_diff_pts_pg_to_date",
    "home_win_pct_to_date":"away_home_win_pct_to_date",
    "road_win_pct_to_date":"away_road_win_pct_to_date",
})

# Enforce key dtypes again post-rename and drop incomplete keys
for df, abbr_col in [(home_tf, "home_abbr"), (away_tf, "away_abbr")]:
    df[abbr_col] = df[abbr_col].astype(str).map(normalize_abbr)
    df["season"] = df["season"].astype(str).map(normalize_season_str)
    df["GAME_ID"] = df["GAME_ID"].astype(str)
    df.dropna(subset=[abbr_col, "season", "GAME_ID"], inplace=True)

# Merge home/away features into base context
ctx = bt_core.merge(
    home_tf[[
        "home_abbr","season","GAME_ID",
        "home_rest_days","home_b2b",
        "home_recent_margin","home_recent_win_pct",
        "home_pts_pg_to_date","home_opp_pts_pg_to_date","home_diff_pts_pg_to_date",
        "home_home_win_pct_to_date","home_road_win_pct_to_date"
    ]],
    on=["home_abbr","season","GAME_ID"],
    how="left"
)

ctx = ctx.merge(
    away_tf[[
        "away_abbr","season","GAME_ID",
        "away_rest_days","away_b2b",
        "away_recent_margin","away_recent_win_pct",
        "away_pts_pg_to_date","away_opp_pts_pg_to_date","away_diff_pts_pg_to_date",
        "away_home_win_pct_to_date","away_road_win_pct_to_date"
    ]],
    on=["away_abbr","season","GAME_ID"],
    how="left"
)

# Derived features
ctx["rest_diff"] = ctx["home_rest_days"] - ctx["away_rest_days"]


ctx["diff_recent_margin"] = ctx["home_recent_margin"] - ctx["away_recent_margin"]
ctx["diff_recent_win_pct"] = ctx["home_recent_win_pct"] - ctx["away_recent_win_pct"]
ctx["diff_pts_pg_to_date"] = ctx["home_diff_pts_pg_to_date"] - ctx["away_diff_pts_pg_to_date"]

# Home home-win% minus away road-win%
ctx["split_win_pct_delta"] = ctx["home_home_win_pct_to_date"] - ctx["away_road_win_pct_to_date"]

# Final selection, standardize, save
required_cols = ["season","GAME_ID","home_abbr","away_abbr","home_win"]
feature_cols = [
    "home_b2b","away_b2b","home_rest_days","away_rest_days","rest_diff",
    "diff_recent_margin","diff_recent_win_pct",
    "diff_pts_pg_to_date","split_win_pct_delta"
]

# Ensure feature columns exist
for c in feature_cols:
    if c not in ctx.columns:
        ctx[c] = np.nan

bt_aug = ctx.copy()
bt_aug["home_win"] = pd.to_numeric(bt_aug["home_win"], errors="coerce").astype("Int64")
bt_aug = bt_aug[bt_aug["home_win"].isin([0,1])].copy()

bt_aug["season"] = bt_aug["season"].astype(str).map(normalize_season_str)
bt_aug["home_abbr"] = bt_aug["home_abbr"].astype(str).map(normalize_abbr)
bt_aug["away_abbr"] = bt_aug["away_abbr"].astype(str).map(normalize_abbr)
bt_aug["GAME_ID"] = bt_aug["GAME_ID"].astype(str)

bt_augmented = bt_aug[required_cols + feature_cols].dropna(subset=required_cols).copy()
bt_augmented = bt_augmented.drop_duplicates(subset=["GAME_ID"]).reset_index(drop=True)

bt_augmented.to_csv("./data/bt_games_with_context.csv", index=False)
print("Saved ./data/bt_games_with_context.csv")
print("Rows:", len(bt_augmented))
print("Columns:", bt_augmented.columns.tolist())

Saved ./data/bt_games_with_context.csv
Rows: 15093
Columns: ['season', 'GAME_ID', 'home_abbr', 'away_abbr', 'home_win', 'home_b2b', 'away_b2b', 'home_rest_days', 'away_rest_days', 'rest_diff', 'diff_recent_margin', 'diff_recent_win_pct', 'diff_pts_pg_to_date', 'split_win_pct_delta']


### Null value Processing

#### 1. Train Test Valid Split & Parse Season Year

In [16]:
cont_cols = ["home_rest_days","rest_diff","diff_recent_margin","diff_recent_win_pct","diff_pts_pg_to_date","split_win_pct_delta",]
bin_col = ["home_b2b"]

def parse_season_start_year(season_str):
    """
    Extracts the first 4-digit year from a season label like '2014-15' or '2014–15'.
    Returns np.nan if not found.
    """
    if pd.isna(season_str):
        return np.nan
    s = str(season_str).strip()
    s = s.replace("–", "-").replace("—", "-")
    m = re.match(r"^\s*(\d{4})\s*-\s*\d{2,4}\s*$", s)
    if m:
        return int(m.group(1))
    # Fallback: first 4-digit year anywhere
    m2 = re.search(r"(\d{4})", s)
    return int(m2.group(1)) if m2 else np.nan

if "season_start_year" not in bt_augmented.columns:
    bt_augmented["season_start_year"] = bt_augmented["season"].apply(parse_season_start_year)

def assign_split(season_start_year):
    if 2014 <= season_start_year <= 2020:
        return "train"
    if 2021 <= season_start_year <= 2023:
        return "test"
    if 2024 <= season_start_year <= 2025:
        return "valid"

bt_augmented["split"] = bt_augmented["season_start_year"].apply(assign_split)

#### 2. Fit Medians

In [17]:
clean_df = bt_augmented.copy()

cont_cols = [c for c in cont_cols if c in clean_df.columns]

# Compute train medians for continuous cols
train_mask = clean_df["split"] == "train"
train_medians = {c: clean_df.loc[train_mask, c].median(skipna=True) for c in cont_cols}

# Fill continuous with train medians
for c in cont_cols:
    clean_df[c] = clean_df[c].fillna(train_medians[c])

# Fill binary with zeros (and keep numeric type)
clean_df[bin_col] = clean_df[bin_col].fillna(0).astype(float)

clean_df.to_csv("./data/bt_games_cleaned.csv",index=False)

----

### Classify Preseason, Regular, All-Star or Playoff games

In [18]:
# Determine corresponding game belongs to which type
def identify_game_type(data_name, output_name):
    df = pd.read_csv(f"data/{data_name}.csv")
    df["Game_Type"] = df["GAME_ID"].astype(str).str[0].map({
        "1": "preseason",
        "2": "regular",
        "3": "allstar",
        "4": "playoff",
        "5": "playin",
        "6": "inseason"
    }).fillna("unknown")
    df.to_csv(f"data/{output_name}.csv", index=False)
    print(f"File saved as data/{output_name}.csv")

identify_game_type("bt_games_cleaned","bt_games_cleaned")

File saved as data/bt_games_cleaned.csv


In [19]:
df = pd.read_csv("./data/bt_games_cleaned.csv")
df_filtered = df[df["Game_Type"].isin(["regular", "playoff", "playin", "inseason"])]
df_filtered.to_csv("./data/bt_final_cleaned.csv", index=False)